In [62]:
from pathlib import Path
import xml.etree.ElementTree as ET
from open_ephys import analysis as oea
import xarray as xr
import numpy as np
import pandas as pd

TODOs:
- `Session` should be able to take `Path` objects, not just `str`
- `RecordNode` objects should have a way to read `settings.xml`

In [47]:
acq_dir = Path("/Volumes/neuropixel_archive/tetrode_data/2026-05-19_18-09-51")
assert acq_dir.exists()

session = oea.Session(str(acq_dir))
print(session)


Open Ephys Recording Session Object
Directory: /Volumes/neuropixel_archive/tetrode_data/2026-05-19_18-09-51

<object>.recordnodes:
  Index 0: Record Node 101 (binary format)



In [48]:
node = session.recordnodes[0]

settings_file = Path(node.directory) / "settings.xml"
assert settings_file.exists()

tree = ET.parse(settings_file)
root = tree.getroot()

channel_map = next(
    p for p in root.findall("./SIGNALCHAIN/PROCESSOR")
    if p.get("name") == "Channel Map"
)
stream = channel_map.find("./CUSTOM_PARAMETERS/STREAM")

indices = [int(ch.get("index")) for ch in stream.findall("CH")]
enabled = [int(ch.get("enabled")) for ch in stream.findall("CH")]

print(f"{len(indices)} channel map found in node settings.xml:")
print("indices:", indices)
print("enabled:", enabled)

64 channel map found in node settings.xml:
indices: [39, 37, 35, 33, 47, 45, 43, 41, 55, 53, 51, 49, 57, 63, 61, 59, 62, 60, 58, 56, 54, 52, 50, 48, 46, 44, 42, 40, 38, 36, 34, 32, 24, 26, 28, 30, 16, 18, 20, 22, 8, 10, 12, 14, 0, 2, 4, 6, 3, 5, 7, 1, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31]
enabled: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [49]:
tt2ch_ixs = {i: indices[i*4:i*4+4] for i in range(len(indices) // 4)}
tt2ch_ixs

{0: [39, 37, 35, 33],
 1: [47, 45, 43, 41],
 2: [55, 53, 51, 49],
 3: [57, 63, 61, 59],
 4: [62, 60, 58, 56],
 5: [54, 52, 50, 48],
 6: [46, 44, 42, 40],
 7: [38, 36, 34, 32],
 8: [24, 26, 28, 30],
 9: [16, 18, 20, 22],
 10: [8, 10, 12, 14],
 11: [0, 2, 4, 6],
 12: [3, 5, 7, 1],
 13: [9, 11, 13, 15],
 14: [17, 19, 21, 23],
 15: [25, 27, 29, 31]}

In [50]:
tt2ch_names = {i+1: [i*4+1, i*4+2, i*4+3, i*4+4] for i in range(len(indices) // 4)}
tt2ch_names

{1: [1, 2, 3, 4],
 2: [5, 6, 7, 8],
 3: [9, 10, 11, 12],
 4: [13, 14, 15, 16],
 5: [17, 18, 19, 20],
 6: [21, 22, 23, 24],
 7: [25, 26, 27, 28],
 8: [29, 30, 31, 32],
 9: [33, 34, 35, 36],
 10: [37, 38, 39, 40],
 11: [41, 42, 43, 44],
 12: [45, 46, 47, 48],
 13: [49, 50, 51, 52],
 14: [53, 54, 55, 56],
 15: [57, 58, 59, 60],
 16: [61, 62, 63, 64]}

In [51]:
print(f"{len(node.recordings)} unique (experiment, recording) pairs found in node: ")
for pair_ix, rec in enumerate(node.recordings):
    print(f"  {pair_ix}: ({rec.experiment_index}, {rec.recording_index})")

3 unique (experiment, recording) pairs found in node: 
  0: (0, 0)
  1: (0, 1)
  2: (0, 2)


In [52]:
pair_ix = 0
rec = node.recordings[pair_ix]
print(rec)

Open Ephys GUI Recording
ID: 0x79ff8d507570
Format: Binary
Directory: /Volumes/neuropixel_archive/tetrode_data/2026-05-19_18-09-51/Record Node 101/experiment1/recording1
Experiment Index: 0
Recording Index: 0


In [53]:
print(f"{len(rec.continuous)} continuous streams found in recording: ")
keys = list(rec.continuous.keys())
for i in range(len(keys) // 2):
    print(f"  {i}: {keys[i + len(keys) // 2]}")

2 continuous streams found in recording: 
  0: acquisition_board
  1: memory_usage


In [54]:
stream_name = "acquisition_board"

for meta in rec.info["continuous"]:
    if meta["stream_name"] == stream_name:
        fs = meta["sample_rate"]

cont = rec.continuous[stream_name]
ns, nc = cont.samples.shape
print(f"{stream_name} stream has {ns} samples and {nc} channels at {fs} Hz")

acquisition_board stream has 1815085 samples and 64 channels at 30000.0 Hz


In [55]:
selected_tetrodes = [1, 3, 6, 14, 15, 16]

Loading strategy: do a single `get_samples` call with all selected Open Ephys
channel indices in the final target order. `cont.samples` is a memory-mapped
`(n_samples, n_channels)` array; each `get_samples` call performs fancy
indexing that materializes one fresh ndarray. One big call is cheaper than
many small ones (fewer Python/IO round-trips, one allocation, no later
concat-copy), and slicing into per-tetrode views afterward is free because
xarray `isel(..., slice)` returns views into the same buffer.

In [61]:
# Build channel ordering: tetrode-major, channel-minor, in requested tetrode order.
oe_ixs, ch_names, tt_labels = [], [], []
for tt in selected_tetrodes:
    oe_ixs.extend(tt2ch_ixs[tt - 1])
    ch_names.extend(tt2ch_names[tt])
    tt_labels.extend([tt] * 4)

# Single contiguous read in the desired channel order.
samples = cont.get_samples(
    start_sample_index=0,
    end_sample_index=ns,
    selected_channels=oe_ixs,
)  # shape: (n_samples, n_selected_channels)

da = xr.DataArray(
    samples,
    dims=("time", "channel"),
    coords={
        "time": ("time", cont.timestamps),
        "channel": ("channel", ch_names),
        "tetrode": ("channel", tt_labels),
        "oe_index": ("channel", oe_ixs),
    },
    attrs={"sample_rate": fs, "units": "microvolts"},
)
da

<xarray.DataArray (time: 1815085, channel: 24)> Size: 348MB
array([[ 65.71499757,  62.98499767,  61.61999772, ..., 147.80999454,
        158.33999415, 157.75499418],
       [ 54.59999798,  58.30499785,  56.35499792, ..., 143.90999469,
        150.53999444, 146.2499946 ],
       [ 42.11999844,  54.40499799,  44.65499835, ..., 148.58999451,
        165.55499389, 164.18999394],
       ...,
       [-49.33499818, -42.31499844, -60.05999778, ...,  -9.94499963,
          3.11999988,   8.96999967],
       [-43.67999839, -30.22499888, -45.23999833, ...,   8.1899997 ,
         10.9199996 ,  26.71499901],
       [-29.05499893, -42.70499842, -48.55499821, ...,   9.16499966,
          7.40999973,  23.98499911]], shape=(1815085, 24))
Coordinates:
  * time      (time) float64 15MB 18.67 18.67 18.67 18.67 ... 79.17 79.17 79.17
  * channel   (channel) int64 192B 1 2 3 4 9 10 11 12 ... 58 59 60 61 62 63 64
    tetrode   (channel) int64 192B 1 1 1 1 3 3 3 3 6 ... 15 15 15 15 16 16 16 16
    oe_index  (channel) int64 192B 39 37 35 33 55 53 51 ... 19 21 23 25 27 29 31
Attributes:
    sample_rate:  30000.0
    units:        microvolts

In [63]:
midx = pd.MultiIndex.from_arrays(
    [tt_labels, ch_names], names=("tetrode", "channel")
)
da2 = xr.DataArray(
    samples,
    dims=("time", "electrode"),
    coords=xr.Coordinates.from_pandas_multiindex(midx, "electrode"),
    attrs={"sample_rate": fs, "units": "microvolts"},
).assign_coords(time=("time", cont.timestamps))
da2

<xarray.DataArray (time: 1815085, electrode: 24)> Size: 348MB
array([[ 65.71499757,  62.98499767,  61.61999772, ..., 147.80999454,
        158.33999415, 157.75499418],
       [ 54.59999798,  58.30499785,  56.35499792, ..., 143.90999469,
        150.53999444, 146.2499946 ],
       [ 42.11999844,  54.40499799,  44.65499835, ..., 148.58999451,
        165.55499389, 164.18999394],
       ...,
       [-49.33499818, -42.31499844, -60.05999778, ...,  -9.94499963,
          3.11999988,   8.96999967],
       [-43.67999839, -30.22499888, -45.23999833, ...,   8.1899997 ,
         10.9199996 ,  26.71499901],
       [-29.05499893, -42.70499842, -48.55499821, ...,   9.16499966,
          7.40999973,  23.98499911]], shape=(1815085, 24))
Coordinates:
  * time       (time) float64 15MB 18.67 18.67 18.67 18.67 ... 79.17 79.17 79.17
  * electrode  (electrode) object 192B MultiIndex
  * tetrode    (electrode) int64 192B 1 1 1 1 3 3 3 3 ... 15 15 15 16 16 16 16
  * channel    (electrode) int64 192B 1 2 3 4 9 10 11 ... 58 59 60 61 62 63 64
Attributes:
    sample_rate:  30000.0
    units:        microvolts

Dimension-order trade-offs:

- `(time, tetrode, channel)` — what we use below. Reshape is a view, time stays
  leading (consistent with `da`/`da2`), and `isel(tetrode=i)` returns a
  contiguous `(time, channel)` block — ideal for per-tetrode spike detection
  or referencing one tetrode's 4 channels against each other. Per-channel time
  series are strided, so single-channel filtering touches non-contiguous
  memory.
- `(tetrode, channel, time)` — per-channel time series are contiguous, which
  is the best layout for per-channel temporal operations (filtering, FFT,
  thresholding). But it requires a transpose copy of the full 348 MB buffer
  on construction, and breaks the workspace convention of time-leading arrays.

In [67]:
n_tt = len(selected_tetrodes)
samples_3d = samples.reshape(ns, n_tt, 4)  # view, no copy

# Channel name within a tetrode is just the within-tetrode index (1..4); the
# global channel name and OE index vary per (tetrode, channel) pair, so they
# become 2-D coords.
tt_names_2d = np.array(tt_labels, dtype=np.int64).reshape(n_tt, 4)
ch_names_2d = np.array(ch_names, dtype=np.int64).reshape(n_tt, 4)
oe_ixs_2d = np.array(oe_ixs, dtype=np.int64).reshape(n_tt, 4)

da3 = xr.DataArray(
    samples_3d,
    dims=("time", "tetrode", "channel"),
    coords={
        "time": ("time", cont.timestamps),
        "tetrode": ("tetrode", list(range(n_tt))),
        "channel": ("channel", list(range(4))),
        "tetrode_name": (("tetrode", "channel"), tt_names_2d),
        "channel_name": (("tetrode", "channel"), ch_names_2d),
        "oe_index": (("tetrode", "channel"), oe_ixs_2d),
    },
    attrs={"sample_rate": fs, "units": "microvolts"},
)
da3

<xarray.DataArray (time: 1815085, tetrode: 6, channel: 4)> Size: 348MB
array([[[  65.71499757,   62.98499767,   61.61999772,   54.79499798],
        [ 156.19499423,  128.30999526,  106.66499606,  143.5199947 ],
        [   4.28999984,    2.33999991,   -6.43499976,   -2.14499992],
        [  38.0249986 ,   55.57499795,   62.98499767,   62.98499767],
        [  14.81999945,   36.65999865,   24.95999908,    0.38999999],
        [ 174.13499357,  147.80999454,  158.33999415,  157.75499418]],

       [[  54.59999798,   58.30499785,   56.35499792,   49.91999816],
        [ 153.07499435,  133.57499507,  106.66499606,  148.19999453],
        [   0.97499996,    4.87499982,   -8.38499969,    6.04499978],
        [  36.46499865,   53.42999803,   57.32999788,   67.07999752],
        [  12.86999952,   30.80999886,   23.20499914,    6.04499978],
        [ 162.82499399,  143.90999469,  150.53999444,  146.2499946 ]],

       [[  42.11999844,   54.40499799,   44.65499835,   54.01499801],
        [ 156.77999421,  129.67499521,  116.99999568,  129.47999522],
        [ -10.7249996 ,    1.16999996,  -15.01499945,    2.33999991],
        [  47.57999824,   49.33499818,   60.05999778,   59.86499779],
        [   2.7299999 ,   30.61499887,   29.63999891,   12.86999952],
        [ 168.67499377,  148.58999451,  165.55499389,  164.18999394]],
...
        [ -13.6499995 ,   -7.40999973,  -34.31999873,  -10.52999961],
        [ -69.61499743,  -79.55999706,  -90.28499667,  -67.85999749],
        [-121.28999552, -113.09999582, -115.63499573, -114.26999578],
        [ -81.50999699, -108.224996  ,  -90.08999667,  -62.59499769],
        [  -3.31499988,   -9.94499963,    3.11999988,    8.96999967]],

       [[ -43.67999839,  -30.22499888,  -45.23999833,  -48.16499822],
        [  -6.62999976,    0.77999997,  -27.49499898,   -2.7299999 ],
        [ -61.03499775,  -44.65499835,  -75.26999722,  -49.52999817],
        [-117.58499566, -108.02999601,  -94.7699965 , -111.1499959 ],
        [ -85.79999683,  -83.65499691,  -77.60999713,  -51.08999811],
        [   5.06999981,    8.1899997 ,   10.9199996 ,   26.71499901]],

       [[ -29.05499893,  -42.70499842,  -48.55499821,  -38.0249986 ],
        [ -10.33499962,   -4.28999984,  -15.59999942,   -0.97499996],
        [ -63.17999767,  -57.52499788,  -70.00499742,  -51.4799981 ],
        [ -92.23499659, -100.42499629,  -92.23499659, -101.20499626],
        [ -75.26999722,  -67.27499752,  -84.82499687,  -56.35499792],
        [  16.57499939,    9.16499966,    7.40999973,   23.98499911]]],
      shape=(1815085, 6, 4))
Coordinates:
  * time          (time) float64 15MB 18.67 18.67 18.67 ... 79.17 79.17 79.17
  * tetrode       (tetrode) int64 48B 0 1 2 3 4 5
  * channel       (channel) int64 32B 0 1 2 3
    tetrode_name  (tetrode, channel) int64 192B 1 1 1 1 3 3 ... 15 16 16 16 16
    channel_name  (tetrode, channel) int64 192B 1 2 3 4 9 10 ... 60 61 62 63 64
    oe_index      (tetrode, channel) int64 192B 39 37 35 33 55 ... 25 27 29 31
Attributes:
    sample_rate:  30000.0
    units:        microvolts